# Project 2: Model Analysis Project

## Table of contents <a id='toc0_'></a>
- [Question 1: A Consumer with Two Nests](#toc1_)
    - [Question 1.1: The problem in nested budget shares](#toc1_1_)
    - [Question 1.2: Calibration](#toc1_2_)
    - [Question 1.3: How do you know your answer is right?](#toc1_3_)
- [Question 2: Solving the Model Numerically](#toc2_)
    - [Question 2.1: A Two-dimensional Grid Search](#toc2_1_)
    - [Question 2.2: L-BFGS-B](#toc2_2_)
- [Question 3: Relative prices and demand](#toc3_)
- [Question 4: Lump-sum taxes and product taxes](#toc4_)
- [Question 5: Extensition](#toc5_)

We import the nessesary packages and the two python files given in the project description:

In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
from scipy import optimize
from types import SimpleNamespace

# plotting
import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# The classes
from Consumer import ConsumerClass
from Government import GovernmentClass
from Extension import PigouvianGovernmentClass

## Question 1: A Consumer with Two Nests <a id='toc1_'></a>

### Question 1.1: The problem in nested budget shares <a id='toc1_1_'></a>

To be able to start the project, we fill out the pass arguments in consumer.py, based on the model description. We have clearly marked in consumer.py where we change the code.

In [ ]:
# we check whether the code runs smoothly after we have filled it in
#Print the parameters
model=ConsumerClass()
print(model)

#We know that with a CES function, inserting x1=x2=x3=1 means that utility should also be 1, as the function collapses:
assert np.isclose(model.utility(1.0,1.0,1.0), 1.0), 'utility(1,1,1) should be 1'
print(' \n utility sanity check passed')

#Could also add a check on whether corner solutions are stable??

So we find that the code we have inserted in consumption.py runs smoothly and holds the basic values that we expected

### Question 1.2: Calibration <a id='toc1_2_'></a>

We now implement the two calibrations that we will use in the following questions, where only the substitution between the modes of transport differ (complements or substitutes respectively). To do this, we let model_comp be where $sigma_B=0.4$, so complements, and let model_sub have the value of $sigma_B=3.0$ so bus and train are substitutes.

In [ ]:
model_comp = ConsumerClass()
model_sub  = ConsumerClass(par={'sigma_B':3.0})

print(model_comp)
print(model_sub)

### Question 1.3: How do you know your answer is right? <a id='toc1_3_'></a>

#### Question 1.3.1

As we cannot check the solution for the model, we check whether the solutions are possible, meaning that the budget shares sum to 1 and are all positive.

In [ ]:
#First, check if the solution is possible
for model in (model_comp, model_sub):
    sol = model.solve(do_print=False)
    assert 0 < sol.s1 < 1 and 0 < sol.s2 and 0 < sol.s3
    assert np.isclose(sol.s1+sol.s2+sol.s3, 1.0), 'the budget shares do not sum to one'

print('the answer is possible')

#### Question 1.3.2

To check the answer, we compare the results we get from the model when solved using optimize.minimize and when we solve the model numerically in section 2. Here, we have inserted the grid-solution from section 2:

In [ ]:
for name, model in models.items():
    sol_grid = model.solve_grid(N=500, do_print=False)
    sol_min  = model.solve(do_print=False)

    assert np.isclose(sol_grid.s1, sol_min.s1, atol=1e-2), f'{name}: s1 does not agree between methods'
    assert np.isclose(sol_grid.w,  sol_min.w,  atol=1e-2), f'{name}: w does not agree between methods'
    assert np.isclose(sol_grid.u,  sol_min.u,  atol=1e-3), f'{name}: u does not agree between methods'

    print(f'{name}: grid search and L-BFGS-B agree '
          f'(Δs1={abs(sol_grid.s1-sol_min.s1):.5f}, '
          f'Δw={abs(sol_grid.w-sol_min.w):.5f}, '
          f'Δu={abs(sol_grid.u-sol_min.u):.6f})')

## Question 2: Solving the Model Numerically <a id='toc2_'></a>

### Question 2.1: A Two-dimensional Grid Search <a id='toc2_1_'></a>

#### Question 2.1.1

We make the grid of the N values in the consumer.py file in solve_grid - we now report the values the weigths and the three implied budget shares for each calibration.

In [ ]:
print('when transport are complements:')
sol_grid_comp = model_comp.solve_grid()
print('when transport are substitutes:')
sol_grid_sub  = model_sub.solve_grid()

So we see that when the budget shares for train and bus are more equal when the modes of transport are complements, whereas when they are substitutes, more are spent on bus trips, and results in a higher utility for the consumer.

#### Question 2.1.2

We now plot the utility over the nested shares in a 3D plot and a contour plot with the solution shown for both calibrations:

In [ ]:
from matplotlib import cm # import the colormap
#Define the figure, so only have to write it out once:
#Because we need the contour plot for 2.2.2, we write it as a seperate function, then call it:

def plot_contour(sol, title, ax=None, path=None): #last two calls needed for 2.2.2
    if ax is None: #so the plot works for 2.1.2, without the ax call
        fig, ax = plt.subplots(figsize=(6.5,5.5))
    else:
        fig = ax.figure

    ax.contourf(sol.s1_grid, sol.w_grid, sol.u_grid, levels=30)
    ax.plot(sol.s1, sol.w, 'o', color='red', ms=10, label='solution')

    if path is not None:
        ax.plot(path[:,0], path[:,1], '-o', color='white', ms=4, lw=1.5, label='convergence path')

    ax.set_title(f'Contour plot, with the solution marked')
    ax.set_xlabel('$s_1$')
    ax.set_ylabel('$w$')
    ax.legend()

    return fig, ax

def plot_utility_surf(sol, title):
    fig = plt.figure(figsize=(13,5.5))

    #The 3d plot:
    ax1 = fig.add_subplot(1,2,1,projection='3d')
    surf = ax1.plot_surface(sol.s1_grid, sol.w_grid, sol.u_grid, cmap='viridis') #make sure the colourmap matches the utility values
    fig.colorbar(surf, ax=ax1, shrink=0.6)
    # labels and titles:
    fig.suptitle(title, fontsize=16, fontweight='bold')
    ax1.set_title('Utility over the nested shares')
    ax1.set_xlabel('$s_1$')
    ax1.set_ylabel('$w$')
    ax1.set_zlabel('$u$')
    #change the direction of the y-axis and where the z-axis is placed
    ax1.view_init(azim=-135,elev=30) 
    ax1.set_box_aspect([4,4,3],zoom=0.8)

    #and now the contour plot:
    ax2=fig.add_subplot(1,2,2)
    plot_contour(sol, title, ax=ax2)


    fig.tight_layout(pad=0.1)


plot_utility_surf(sol_grid_comp, 'Complements')

plot_utility_surf(sol_grid_sub, 'Substitutes')

So we see that when means of transportation are substitutes, the weight on the bus, w, is higher compared to when it is complements, as here the consumer wants to spread the weight on both trains and busses. We also see that the utility surface seem to be flatter for subsitutes, meaning that a wider array of w-values can lead to almost the same utility-value compared to when it is complements, where the utility surface is more pointed.

#### Question 2.1.3

We now test how sensitive the grid solving method is to the value of N, meaning how fine the grid is. The finer the grid, the more evaluations of u.

In [ ]:
#for the 4 N-values and the two model calibrations:
N_values=[50, 100, 500, 1000]
models={'Complements': model_comp, 'Substitutes': model_sub}

results={}
for name, model in models.items():
    for N in N_values:
        opt=model.solve_grid(N=N, do_print=False)
        results[(name, N)]= opt
        print(f'{name}, N={N:4d}: s1={opt.s1:.4f}, w={opt.w:.4f}, u={opt.u:.6f}, evaluations={N*N}')
    print() 

#To see how much the answer moves, we look at the differences in s1, w and u, every time N becomes larger:
print('Changes in the nested budget share and the utility with a finer grid search')
for name in models:
    print(name)
    for N_prev, N in zip(N_values, N_values[1:]):
        opt_prev = results[(name,N_prev)]
        opt = results[(name,N)]
        print(f'  N={N:4d}: Δs1={abs(opt.s1-opt_prev.s1):.5f}, Δw={abs(opt.w-opt_prev.w):.5f}, Δu={abs(opt.u-opt_prev.u):.6f}')


We see that the number of evaluations of u grows quadratically with N, since the grid search evaluates every point on an N×N grid. With a finer grid, the answer does move, and the answer improves smoothly: it becomes more accurate as the grid gets finer, but only slightly once N is already large. This higher accuracy comes at a high computational cost, since the number of evaluations grows rapidly with N. Comparing the two calibrations, the improvement in u with a finer grid is small for both, but the improvement in w converges more slowly for substitutes than for complements. As we saw in 2.1.2, many w-values give almost the same u-value under substitutes (flatter utility surface), so the grid search converges more slowly onto the exact best w — even though the resulting utility is still accurate.

#### Question 2.1.4

All in all, we find that for both calibrations the surface is steep along the s1-direction (food), so there is a high cost to utility when moving food away from optimum. However, in the w-direction, the two different calibrations are quite different where the surface is steep when means of transportation are complements, indicating a peak around the optimum. For substitutes, the surface is more flat in the w-direction, indication a more flat plateau around optimum. Therefore we expect that it is harder for an optimizer to find the answer when busses and trains are substitutes, as the utility only changes slightly for different w-values when close to optimum, making it harder to find out what direction improves utility. This was shown very clearly in 2.1.3, where the improvement in w converged more slowly for subsitutes than for complements as the grid was refined.

### Question 2.2: L-BFGS-B <a id='toc2_2_'></a>

#### Question 2.2.1

We implement the L-BFGS-B solution in the consumer.py file, calling it solve. We compare the solution, the number of function evaluations and the running time with the grid search, that we made in 2.1    

In [ ]:
# We import time, and run the solutions for both model calibration.
import time
sol_min_comp = model_comp.solve(do_print=False)
sol_min_sub  = model_sub.solve(do_print=False)
models = {'Complements': model_comp, 'Substitutes': model_sub}

for name, model in models.items():
    print(f'--{name}--')

    # grid search
    t0 = time.perf_counter()
    sol_grid = model.solve_grid(do_print=False) 
    t_grid = time.perf_counter() - t0

    # L-BFGS-B
    t0 = time.perf_counter()
    sol_min = model.solve(do_print=False)
    t_min = time.perf_counter() - t0


    print(f'  grid search: s1={sol_grid.s1:.4f}, w={sol_grid.w:.4f}, u={sol_grid.u:.6f}, '
          f'evaluations=40000, time={t_grid:.4f}s') # evaluations are 40.000 since N=200 if we do not change it
    print(f'  L-BFGS-B:    s1={sol_min.s1:.4f}, w={sol_min.w:.4f}, u={sol_min.u:.6f}, '
          f'evaluations={sol_min.res.nfev}, time={t_min:.4f}s')
    print()



So we find that the methods find slightly different solutions (but agree at least on the first two decimals), and that the grid search takes slightly longer, with a much higher count of evaluations. So the two methods do converge towards the same answer, but they do not agree. This is because the grid is 200 x 200, and therefore perhaps not fine enough. We also see that L-BFGS-B needs only few evaluations, because it uses a quasi-Newton method with bounds - this is a lot more targeted than the grid search.

#### Question 2.2.2

We now record the convergence path, seeing how the quasi-Newton method with bounds find the answer. We plot this in the contour plot we made in 2.1.2 but also plot how far each of the iterations are from the solving-point on a log-scaled iteration:

In [ ]:
#Define a figure with both plots:
def plot_convergence_full(sol_grid, sol_min, title):

    fig = plt.figure(figsize=(13,5.5))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # contour plot with the solve path overlaid
    ax1 = fig.add_subplot(1,2,1)
    plot_contour(sol_grid, title, ax=ax1, path=sol_min.path)

    # distance from the final point, using a log scale
    ax2 = fig.add_subplot(1,2,2)
    path = sol_min.path
    final = path[-1] # so the last value in path
    distances = np.linalg.norm(path - final, axis=1)

    ax2.plot(distances, '-o', ms=4)
    ax2.set_yscale('log')
    ax2.set_title(f'Distance from final point')
    ax2.set_xlabel('iteration $k$')
    ax2.set_ylabel('distance (log scale)')

    fig.tight_layout(pad=0.1)

plot_convergence_full(sol_grid_comp, sol_min_comp, 'Complements')

plot_convergence_full(sol_grid_sub, sol_min_sub, 'Substitutes')

So we see that convergence is slower for substitutes in the beginning, and then the distance gets smaller very quickly. For complements, the convergence is quicker. The method is for both especially quick after the first iteration. We also see that the method is closer to the final answer from the first iteration.

#### Question 2.2.3

We now solve from 6 different starting points by calling solve with different starting point $s_0$. The starting point has for all solved been $[1/2, 1/2]$, so we print that again and now do the four corners plus a random point:

In [ ]:

starts=[(0.5,0.5),(0,0),(0,1),(1,0),(1,1),(0.6,0.8)] #our middle, 4 corners and a random
models = {'Complements': model_comp, 'Substitutes': model_sub}

for name, model in models.items():
    print(f'--{name}--')
    for s0 in starts:
        sol = model.solve(s0=np.array(s0), do_print=False)
        print(f'  s0={s0}: s1={sol.s1:.4f}, w={sol.w:.4f}, u={sol.u:.6f}, '
            f'success={sol.res.success}, iterations={sol.res.nit}')

    print()


So we find that when transport are substitutes, no matter the starting point, the solve finds the same values for s1, w and u. However, it does take more iterations to converge when starting in the corners, compared to starting in the middle. For complements, we find that when starting in the corners, two of them do not find the solution, and stop after 1 iteration. Both of them start form s1=1, and a possible explanation could be that when all income is spent on food, the equation for s2 and s3 becomes 0 (3), and therefore the objective is exactly flat in the corner. This is combined with the CES function raising it to a negative power as $p_B=1-1/0.4=-1.5$, resulting in extreme intermediate values that make L-BFGS-B's numerical gradient estimate unreliable and triggers premature convergence.

#### Question 2.2.4

We now vary the settings of ftol (how much utility improves/changes after each generation, small enough changes must mean close to optimum) and gtol (the gradient is small/flat enough, near-zero gradient must be near the optimum):

In [ ]:
ftol_val=[1e-4, 1e-8, 1e-12, 1e-16] #make our vectors
gtol_val=[1e-2, 1e-5, 1e-8, 1e-12]

rows = []

for name, model in models.items():
    for ftol in ftol_val:
        sol = model.solve(options={'ftol': ftol}, do_print=False)
        rows.append({'calibration': name, 'setting': 'ftol', 'value': ftol,
                     's1': sol.s1, 'w': sol.w, 'u': sol.u, 'nfev': sol.res.nfev}) #choosing row names
    for gtol in gtol_val:
        sol = model.solve(options={'gtol': gtol}, do_print=False)
        rows.append({'calibration': name, 'setting': 'gtol', 'value': gtol,
                     's1': sol.s1, 'w': sol.w, 'u': sol.u, 'nfev': sol.res.nfev})

df = pd.DataFrame(rows)
df.style.format({'value': '{:.0e}', 's1': '{:.6f}', 'w': '{:.6f}',
                  'u': '{:.6f}', 'nfev': '{:.0f}'}) #choose 6 decimals
    

So we see that the lower we set the tolerance for changes in u and the size of the gradient, it does not really change the values for complements, but does increase the function evaluations (nfev) between $1e-04$ and $1e-08$. For substitutes, making the tolerance lower for ftol and gtol does slightly change the value of s1 slightly. Again, we see the function evaluation rise with a lower tolerance. 

#### Question 2.2.5

When we look at the settings, we see that there is very little to gain with a lower tolerance, as it returns very similar utility values, albeit slightly different s1 values. Looking at the function evaluations, they do seem to be stable after $gtol/ftol= 1e-08$, and therefore we would choose this as our tolerance

## Question 3: Relative prices and demand <a id='toc3_'></a>

#### Question 3.3.1

We find the optimal budget shares and quantities across a range of prices for train tickets, p3, whilst holding p1, p2 and I fixed.

In [ ]:
# i. make a range of prices from 0.5 to 3.0
price3_range = np.linspace(0.5,3,50)

# ii. Solving the model when means of transport are compliments across this range of p3's
resultscomp = [] # container for results
for p3 in price3_range:
    model_comp.par.p3 = p3
    sol = model_comp.solve(do_print=False)

    s1, s2, s3 = model_comp.shares(sol.s1, sol.w) # saving optimal shares
    x1, x2, x3 = model_comp.quantities(sol.s1, sol.w) # saving optimal quantities

    resultscomp.append({ # appending these in dictionary "results"
        'p3': p3,
        's1': s1, 's2': s2, 's3': s3,
        'x1': x1, 'x2': x2, 'x3': x3,
    })

df_C = pd.DataFrame(resultscomp)

# ii. Solving the model with substitutes across this range of p3's
resultssub = [] # container for results
for p3 in price3_range:
    model_sub.par.p3 = p3
    sol = model_sub.solve(do_print=False)

    s1, s2, s3 = model_sub.shares(sol.s1, sol.w) # saving optimal shares
    x1, x2, x3 = model_sub.quantities(sol.s1, sol.w) # saving optimal quantities

    resultssub.append({ # appending these in dictionary "results"
        'p3': p3,
        's1': s1, 's2': s2, 's3': s3,
        'x1': x1, 'x2': x2, 'x3': x3,
    })

df_S = pd.DataFrame(resultssub)


# Now we plot the budget shares and quantities

# i. Setting up a figure with 2x2 panels
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Budget shares and quantities demanded when complements')

# ii. Plotting the budget shares across p3-values for model with complements
ax = axes[0,0]
ax.plot(df_C['p3'], df_C['s1'], label='$s_1$ (Food)')
ax.plot(df_C['p3'], df_C['s2'], label='$s_2$ (Bus)')
ax.plot(df_C['p3'], df_C['s3'], label='$s_3$ (Train)')
ax.set_xlabel('$p_3$ (Train price)')
ax.set_ylabel('Budget share')
ax.set_ylim(0,1)
ax.set_title('Complements ($\\sigma_B=0.4$)')

# iii. Plotting the quantities across p3-values for model with complements
ax = axes[1,0]
ax.plot(df_C['p3'], df_C['x1'], label='$x_1$ (Food)')
ax.plot(df_C['p3'], df_C['x2'], label='$x_2$ (Bus)')
ax.plot(df_C['p3'], df_C['x3'], label='$x_3$ (Train)')
ax.set_xlabel('$p_3$ (Train price)')
ax.set_ylabel('Quantity')
ax.set_ylim(0,7)

# iv. Plotting the budget shares across p3-values for model with substitutes
ax = axes[0,1]
ax.plot(df_S['p3'], df_S['s1'], label='$s_1$ (Food)')
ax.plot(df_S['p3'], df_S['s2'], label='$s_2$ (Bus)')
ax.plot(df_S['p3'], df_S['s3'], label='$s_3$ (Train)')
ax.set_xlabel('$p_3$ (Train price)')
ax.set_ylabel('Budget share')
ax.set_ylim(0,1)
ax.set_title(('Substitutes ($\\sigma_B=3$)'))

# v. Plotting the quantities across p3-values for model with substitutes
ax = axes[1,1]
ax.plot(df_S['p3'], df_S['x1'], label='$x_1$ (Food)')
ax.plot(df_S['p3'], df_S['x2'], label='$x_2$ (Bus)')
ax.plot(df_S['p3'], df_S['x3'], label='$x_3$ (Train)')
ax.set_xlabel('$p_3$ (Train price)')
ax.set_ylabel('Quantity')
ax.set_ylim(0,7)


# vi. Fixing legend box to a single positioned at center bottom
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.02))

# vii. Fixing background color for left and right column respectively to highlight that 
# left side panels are from complement model and right side panels are from substitute model
for ax in axes[:, 0]:  # left coloumn
    ax.set_facecolor('#f0f4ff')   

for ax in axes[:, 1]:  # right coloumn
    ax.set_facecolor('#fff4f0')   


fig.tight_layout(rect=[0, 0.06, 1, 1])

fig.text(0.5, -0.02,
         'Note: light blue background = complements ($\\sigma_B=0.4$), '
         'light orange background = substitutes ($\\sigma_B=3$).',
         ha='center', fontsize=10, style='italic')


plt.show()


#### Question 3.3.2

The figure above shows that the budget share of train travel (the green line in the upper panels) *increases* with the price of trains, $p_3$, in the model assuming complements (left panel), while it *decreases* with $p_3$ in the model assuming substitutes (right panel).

This difference arises because the substitution effect dominates the income effect, as explained in more detail in the answer to the question below.

#### Question 3.3.3

As the price of trains, $p_3$, increases, the number of bus trips (orange line in the lower panels) *decreases* in the model assuming complements (left side). Since complementary goods are, by definition, poor substitutes for each other, the consumer does not substantially substitute away from train tickets when their price increases. Instead, the consumer continues to purchase train tickets, as also reflected in the increasing budget share spent on trains in the panel above.

The decrease in bus quantities therefore arises because the income effect dominates the substitution effect. Although the higher price of train tickets creates some incentive to substitute towards bus trips, the consumer has less income available for purchasing bus tickets. Consequently, the income effect outweighs the substitution effect, resulting in an overall decrease in the quantity of bus tickets purchased.

The opposite pattern can be observed in the right-hand panel, where trains and buses are assumed to be substitutes. Here, higher train prices lead to an *increase* in the quantity of bus tickets purchased. While the income effect of higher train prices again reduces the consumer’s purchasing power and therefore pushes towards fewer bus trips, the substitution effect is stronger because trains and buses are highly substitutable. As a result, the substitution effect dominates, leading to an increase in the quantity of bus tickets purchased.

#### Question 3.3.4

From the lower panels, it can be seen that there are very few differences between the models in how the demand for food changes as the train ticket price, $p_3$, rises.

This is because the elasticity of substitution between food and travel, $\sigma_A$, is fixed across the two models, while most of the effect of the price change is absorbed within the transport nest. As the relative price of transportation changes, the quantity of food purchased therefore changes only slightly. The non-zero change in food demand is driven by the income effect. As discussed above, this effect is stronger when buses and trains are complements, causing the quantity of food purchased to fall slightly more in the model with complementary goods. However, the overall effect remains small due to the nested structure of the utility function. Since buses and trains form a separate nested bundle, the effect of the price change is primarily absorbed within this bundle through changes in the quantities of train and bus trips purchased.

Comparing the change in food demand with the change in bus demand further illustrates the importance of the nested structure. Food is outside the transport nest, whereas bus travel is part of it, and the two therefore respond quite differently to the change in the train price. If the elasticity of substitution between food and transport, $\sigma_A$, were larger, a greater share of the price effect would instead be absorbed through changes in food consumption.

## Question 4: Lump-sum taxes and product taxes <a id='toc4_'></a>

#### Question 4.1

We have implimented equation 5 in the `tax_revenue()` and checks the implimentation is correct by comparing it to a by-hand calculation. 

In [ ]:
model_gov = GovernmentClass()
model_gov.set_taxes(T=0.5, tau1=0.1, tau2=0.2, tau3=0.3) #Set the lump-sum tax and the tax of the three goods

opt = model_gov.solve(do_print=False)
R = model_gov.tax_revenue(opt)

# recompute eq. 5 by hand, from the same solution, as an independent check
par = model_gov.par
x1,x2,x3 = model_gov.quantities(opt.s1,opt.w)
R_check = par.T + par.tau1*par.p1_pre*x1 + par.tau2*par.p2_pre*x2 + par.tau3*par.p3_pre*x3

assert np.isclose(R,R_check), f'R={R:.4f} does not match R_check={R_check:.4f}' #We check that the methods agree

print(f'tax_revenue() implements eq. 5:  R = T + tau1*p1_pre*x1 + tau2*p2_pre*x2 + tau3*p3_pre*x3')
print(f'R = {R:.4f}, matching the hand-computed value of {R_check:.4f}')


#### Question 4.2

We look at the revenue as a function of the tax rate when we impliment tax rates on different combinations of goods and the amount of goods that are taxed.

In [ ]:
# Makes the different combinations of goods to be taxed, and the corresponding labels
goods_list = [(1,), (2,), (3,), (2,3), (1,2,3)]
labels     = ['food only', 'bus only', 'train only', 'bus and train', 'all three']

calibrations = {'Complements ($\\sigma_B=0.40$)': {'sigma_B':0.40},
                 'Substitutes ($\\sigma_B=3.00$)': {'sigma_B':3.00}}

tau_vec = np.linspace(0,3,200) #creating the different tax-rates

# Makes the laffer curves for different taxes and different calibrations
fig,axes = plt.subplots(1,2,figsize=(12,6),sharey=True)

for ax,(title,par_update) in zip(axes,calibrations.items()):

    model = GovernmentClass(par=par_update)

    for goods,label in zip(goods_list,labels):
        R_vec = np.array([model.revenue_and_utility(tau,goods=goods)[0] for tau in tau_vec]) #[0] after revenue_and_utility(), drop utility since we only want Revenue
        ax.plot(tau_vec,R_vec,label=label)

    ax.set_xlabel(r'tax rate, $\tau$')
    ax.set_title(title)
    ax.legend()

fig.suptitle('The Laffer Curve for Different Taxes and Different Calibrations')
axes[0].set_ylabel('revenue, $R$')

fig.tight_layout()
plt.show()


Comparing the two panels taxing food alone raises almost identical revenue in both calibrations, since food's substitutability with travel ($\sigma_A=0.80$) is the same in both. This is by construction since the calibrations only differ in $\sigma_B$. For the bus-only and train-only taxes in the complements calibration ($\sigma_B=0.40$), revenue keeps rising steadily across the whole range, because the consumer wants both, and therefore cannot 'escape' the tax. For the substitutes calibration ($\sigma_B=3.00$) shows the same two taxes flattening or turning over much sooner, as the consumer shifts sharply to whichever travel good stays untaxed. Once bus and train are taxed at the same rate, that escape route disappears and the two calibrations converge again, producing far more similar curves than the single-good taxes did. Overall, the complements calibration is the more dependable revenue source at high tax rates, while revenue in the substitutes calibration depends heavily on which good is taxed.

#### Question 4.3

Looking at the plots from question 4.2 only two Laffer Curves have a maxima, both of them is in the substitution calibration and is the scenario when only bus or train is taxed, respectively. We now find the find the revenue-maximizing rate and the largest possible revenue where there is a maxima.

In [ ]:
# Look at the substitution calibration
model_sub = GovernmentClass(par={'sigma_B':3.00})

# The two curves with a visible top (from the 4.2 plot, both well inside [0,3])
tau_bus,   R_bus   = model_sub.max_revenue(goods=(2,), tau_max=3.0)
tau_train, R_train = model_sub.max_revenue(goods=(3,), tau_max=3.0)

print(f'We see that the most we can tax bus rides in the substitutes calibration is')
print(f'{tau_bus:.2f} with a revenue of {R_bus:.2f}, while the most we can tax train')
print(f'rides is {tau_train:.2f} with a revenue of {R_train:.2f}.')


We now examine whether the other Laffer Curves have a maxima or keep raising. 

In [ ]:
# Confirm the other scenarios keep rising
for model,cal_name in [(GovernmentClass(par={'sigma_B':0.40}),'complements'),
                        (GovernmentClass(par={'sigma_B':3.00}),'substitutes')]:
    for goods,label in zip(goods_list,labels):
        if cal_name == 'substitutes' and goods in [(2,),(3,)]:
            continue  # already found above -- these do have a top
        tau,R = model.max_revenue(goods=goods, tau_max=500.0)
        flag = ' (still rising, no maxima found)' if tau == 500.0 else ''
        print(f'{cal_name:12s} | {label:15s}: tau* = {tau:6.2f}, R = {R:8.4f}{flag}')
        print('')

Thus, the only two cases with a maxima are in substitution calibration and for the scenario when only bus or train is taxed, respectively. The taxation rate to generate the most revenue are relatively low, with tau under 1.

#### Question 4.4

We look at the consumer’s utility against the revenue raised, for all five product taxes and for the lump-sum tax, and for both calibration.

In [ ]:
# The tau_vec also decides the grid range. Thus, using the same tau_vec as above, 
# we get a plot with unfished curves for the cases that do not have a maximum in [0,3]. 
#tau_vec = np.linspace(0,3,200) 

fig,axes = plt.subplots(1,2,figsize=(12,5),sharey=True)

for ax,(title,par_update) in zip(axes,calibrations.items()):

    model = GovernmentClass(par=par_update)

    # i. the five product taxes
    R_max = 0.0
    for goods,label in zip(goods_list,labels):
        RU = np.array([model.revenue_and_utility(tau,goods=goods) for tau in tau_vec])
        R_vec,u_vec = RU[:,0],RU[:,1]
        ax.plot(u_vec,R_vec,label=label)
        R_max = max(R_max,R_vec.max())

    # ii. the lump-sum tax, over the same revenue range the product taxes reached
    T_vec = np.linspace(0,R_max,200)
    RU_T = np.array([model.revenue_and_utility_lump_sum(T) for T in T_vec])
    ax.plot(RU_T[:,1],RU_T[:,0],label='lump-sum',linestyle='--',color='black')

    ax.set_xlabel('utility $u$')
    ax.set_title(title)
    ax.legend()

axes[0].set_ylabel('revenue $R$')
fig.tight_layout()
plt.show()


All six curves start from the same point, bottom-right with $R=0$, since $\tau=0$ or $T=0$. Thus, no tax at all yield the identical undistorted baseline utility and zero
revenue regardless of which good is nominally being taxed. Reading each curve from right to left therefore shows that increasing tax means that utility falls and revenue rises as the tax
rate grows from zero. In the complements panel, all five product taxes stay close to the lump-sum frontier along their whole path, only drifting more left when $\tau$ is high. Therefore, whether bus or train is taxed barely when transport are complements. In the substitutes panel this breaks down sharply for bus-only and train-only. Instead of continuing up and to the left, those two curves are concave, since further tax increases now reduce revenue through base erosion whilst utility keeps falling regardless. The remaining substitutes-panel curves stay close to the frontier just as in the complements panel, showing the inefficiency is specific to taxing a single good with an easy substitute. The lump-sum tax and taxing all three goods equally are always cheapest in terms of lost utility and this is calibration-independent.

#### Question 4.5

We look at when the government need $R = 0.20$, i.e. 2 percent of income since $I = 10$.

In [ ]:
R_target = 0.20

for cal_name,par_update in calibrations.items():

    model = GovernmentClass(par=par_update) # Overrides sigme_b to get a fresh consumer/government for this calibration
    rows = [] # Empty list that will collect one (instrument, rate, utility) tuple per row

    for goods,label in zip(goods_list,labels):

        # i. restrict to the efficient (lower) root where a Laffer peak exists
        if cal_name.startswith('Substitutes') and goods in [(2,),(3,)]: # checks whether this is one of the two with a Laffer peak
            tau_peak,_ = model.max_revenue(goods=goods,tau_max=3.0) # Discard the revenue
            bracket = (1e-10,tau_peak)
        else:
            bracket = (1e-10,3.0) # Used to restrict the root-finder to the lower root, if a Laffer peak exists.

        tau = model.find_tax_rate(R_target,goods=goods,bracket=bracket)
        u = np.nan if np.isnan(tau) else model.revenue_and_utility(tau,goods=goods)[1] # utility is index 1
        rows.append((label,tau,u))

    # ii. the lump-sum tax: R = T exactly, no root-finder needed TROR DET ER NOGET JUKS IKKE AT BRUGE HINTET??
    T = R_target
    u_T = model.revenue_and_utility_lump_sum(T)[1] # utility is index 1
    rows.append(('lump-sum',T,u_T))

    df = pd.DataFrame(rows,columns=['instrument','rate (tau or T)','utility'])
    df = df.sort_values('utility',ascending=False).reset_index(drop=True)

    print(cal_name)
    display(df)


In the complements calibration all six instruments cluster tightly, so the ranking barely matters there. In the substitutes calibration, lump-sum and all-three stay tied for cheapest, while train only is clearly the worst instrument and it needs the highest rate of any option ($\tau\approx0.236$, over triple bus-only's $0.070$). Because bus and train are strong substitutes, consumers already lean toward the cheaper bus pre-tax, so a train-only tax hits an already-thin base that erodes fast, forcing a disproportionately large rate and the largest utility loss when raising the same $R=0.20$.


## Question 5: Extension <a id='toc5_'></a>

We extend the model by adding a negative externality to bus trips. The idea is that while trains run on electricity, buses rely on fuel and emit more CO₂ per passenger — so each bus trip imposes a social cost that the consumer does not face when choosing how much to travel by bus. We let $c_2$ denote this external cost per unit of $x_2$, and define social welfare as $W = u - c_2 x_2$: private utility minus the externality.

The government can internalize this cost with a Pigouvian tax $\tau_2$ on bus trips, raising the price the consumer pays from $p_2$ to $(1+\tau_2)p_2$ — the same mechanism used for the product taxes in section 4. We plot revenue $R$, utility $u$, and welfare $W$ across a range of $\tau_2$, and compare the revenue-maximizing and welfare-maximizing tax rates under both calibrations, to see whether a government aiming purely to raise revenue would set a very different rate than one aiming to correct the externality.

In [ ]:
#First, check if the solution is possible
for model in (model_comp, model_sub):
    sol = model.solve(do_print=False)
    assert 0 < sol.s1 < 1 and 0 < sol.s2 and 0 < sol.s3
    assert np.isclose(sol.s1+sol.s2+sol.s3, 1.0), 'the budget shares do not sum to one'

print('the answer is possible')

In [ ]:
c2 = 0.3  # external cost per bus trip, consumer doesn't see this

for cal_name,par_update in calibrations.items():

    par = {**par_update, 'c2':c2}
    model = PigouvianGovernmentClass(par=par) #see Extension.py

    tau_R,R_max = model.max_revenue(goods=(2,), tau_max=3.0)
    tau_W,W_max = model.max_welfare(goods=(2,), tau_max=3.0)

    print(cal_name)
    print(f'  revenue-maximizing tau  = {tau_R:.4f}  (R = {R_max:.4f})')
    print(f'  welfare-maximizing tau  = {tau_W:.4f}  (W = {W_max:.4f})')

tau_vec = np.linspace(0,3,200) # look over the tau-values

fig,axes = plt.subplots(1,2,figsize=(12,5),sharey=False)

for ax,(cal_name,par_update) in zip(axes,calibrations.items()):

    par = {**par_update, 'c2':c2}
    model = PigouvianGovernmentClass(par=par)

    RUW = np.array([model.revenue_utility_welfare(tau,goods=(2,)) for tau in tau_vec])
    R_vec,u_vec,W_vec = RUW[:,0],RUW[:,1],RUW[:,2]

    ax.plot(tau_vec,R_vec,label='revenue $R$')
    ax.plot(tau_vec,u_vec,label='utility $u$')
    ax.plot(tau_vec,W_vec,label='welfare $W = u - c_2 x_2$')

    tau_R,_ = model.max_revenue(goods=(2,), tau_max=3.0)
    tau_W,_ = model.max_welfare(goods=(2,), tau_max=3.0)
    ax.axvline(tau_R,linestyle=':',color='C0',label=r'revenue-max $\tau_2$')
    ax.axvline(tau_W,linestyle=':',color='C2',label=r'welfare-max $\tau_2$')

    ax.set_xlabel(r'tax rate on bus trips $\tau_2$')
    ax.set_title(cal_name)
    ax.legend()

fig.tight_layout()
plt.show()



We find that whether bus and train are complements or substitutes matters greatly for how closely a revenue-maximizing tax aligns with a welfare-maximizing one. Under complements, the welfare-maximizing tax rate is near zero while the revenue-maximizing rate is far higher: since consumers still want both means of transport regardless of price, taxing buses imposes a real welfare cost almost immediately, even as revenue keeps rising with the tax rate because demand for bus trips is always there (until it becomes too expensive). Under substitutes, the two optimal rates sit much closer together for welfare vs. revenue: because consumers can easily switch to trains, taxing buses corrects the externality with comparatively little welfare loss, and this same substitutability also means revenue eventually declines at high tax rates, as the taxed good's consumption largely disappears rather than being paid for at a higher price.

This highlights a general lesson about Pigouvian taxation: the possibility of raising revenue while also correcting an externality depends critically on the substitutability of the taxed good. When substitutes are readily available, a government's revenue and welfare objectives are more closely aligned; when they are not, a policymaker aiming to maximize revenue would set a rate far from what is socially optimal.